# GameTheory-02d-Epistemique-Connaissance-Commune-Python

> **Navigation** : [Notebooks](../) | [Série GameTheory](README.md) | [<< 2-NormalForm (palier parent)](GameTheory-02-NormalForm-Python.ipynb) | [2b - Définitions Lean](GameTheory-02b-Lean-Definitions-Lean.ipynb) | [2c - Dilemme des Voyageurs](GameTheory-02c-Travelers-Dilemma-Python.ipynb)

> **Série** : GameTheory-02d-Epistemique-Connaissance-Commune-Python — accrétion du palier 02 : ce que les joueurs **savent**, avant ce qu'ils choisissent.

## Pourquoi ce carnet

Le palier 02 élimine les stratégies dominées et énumère les supports mixtes. Il le fait sans jamais dire **ce qu'un joueur sait**. Or l'élimination itérée n'est pas une règle de calcul qu'on postule : c'est la conséquence d'une hypothèse sur la connaissance des joueurs, et cette hypothèse a un nom précis — la **connaissance commune de la rationalité**.

Ce carnet construit la machinerie minimale qui rend cette phrase vérifiable par le calcul :

1. un **modèle à partitions** — un ensemble de mondes possibles, une partition par joueur, un opérateur de connaissance ;
2. la **connaissance commune** comme point fixe, distincte de la connaissance mutuelle ;
3. le théorème d'**agreeing to disagree** d'Aumann (1976), démontré ici par énumération exhaustive plutôt que cité, avec son contrôle négatif.

Le fil directeur est un piège de formalisation que le carnet mesure au lieu de l'éviter : « les joueurs sont en désaccord » et « les joueurs savent qu'ils sont en désaccord **sur ces valeurs-là** » sont deux événements différents, et seul le second est impossible sous prior commun.

## 1. Un modèle épistémique : mondes, partitions, connaissance

On se donne un ensemble fini de **mondes** $\Omega$ — chaque monde est une description complète de l'état des choses, y compris ce que chacun sait. L'information du joueur $i$ est une **partition** $P_i$ de $\Omega$ : dans le monde $\omega$, le joueur $i$ ne distingue pas $\omega$ des autres mondes de la même cellule $P_i(\omega)$.

L'**opérateur de connaissance** est alors entièrement déterminé par l'information :

$$K_i(E) = \{\omega \in \Omega : P_i(\omega) \subseteq E\}$$

Un joueur *sait* $E$ quand son information, quelle que soit la vraie situation, ne laisse aucune place à $\lnot E$. La connaissance n'est pas une croyance : elle est vraie par construction ($K_i(E) \subseteq E$), et c'est ce qui lui donne les trois axiomes dits **S5** — véracité, introspection positive ($K_i E \Rightarrow K_i K_i E$) et introspection négative ($\lnot K_i E \Rightarrow K_i \lnot K_i E$).

In [1]:
from itertools import combinations
from dataclasses import dataclass, field

# Les quatre mondes du modele temoin. Chaque monde est un etat complet du monde.
OMEGA = frozenset({0, 1, 2, 3})


@dataclass(frozen=True)
class EpistemicModel:
    """Modele a partitions : un monde fini, une partition par joueur, un prior commun."""

    worlds: frozenset
    partitions: tuple          # un tuple de frozensets par joueur
    prior: dict = field(default_factory=dict)

    def cell(self, player: int, world: int) -> frozenset:
        """Cellule d'information du joueur dans ce monde : ce qu'il ne distingue pas."""
        for block in self.partitions[player]:
            if world in block:
                return block
        raise ValueError(f"monde {world} absent de la partition du joueur {player}")

    def knows(self, player: int, event) -> frozenset:
        """K_i(E) : les mondes ou le joueur i sait que E, parce que sa cellule est incluse dans E."""
        event = frozenset(event)
        return frozenset(w for w in self.worlds if self.cell(player, w) <= event)

    def valid(self) -> bool:
        """Verifie que chaque partition est bien une partition de Omega."""
        return all(
            frozenset().union(*part) == self.worlds
            and sum(len(b) for b in part) == len(self.worlds)
            for part in self.partitions
        )


# Modele temoin : deux joueurs, informations asymetriques.
# Joueur 0 confond 0 et 1, et confond 2 et 3. Joueur 1 distingue 0, confond 1 et 2, distingue 3.
MODEL = EpistemicModel(
    worlds=OMEGA,
    partitions=(
        (frozenset({0, 1}), frozenset({2, 3})),
        (frozenset({0}), frozenset({1, 2}), frozenset({3})),
    ),
    prior={w: 0.25 for w in OMEGA},
)

E = frozenset({0, 1, 2})   # l'evenement "le monde est dans {0, 1, 2}"

print("modele valide        :", MODEL.valid())
print("K_0(E)               :", sorted(MODEL.knows(0, E)))
print("K_1(E)               :", sorted(MODEL.knows(1, E)))

modele valide        : True
K_0(E)               : [0, 1]
K_1(E)               : [0, 1, 2]


### Lecture du résultat

`K_0(E) = {0, 1}` : quand le monde vaut 0 ou 1, le joueur 0 sait que $E$ — sa cellule $\{0,1\}$ est entièrement contenue dans $E$. Aux mondes 2 et 3, sa cellule $\{2,3\}$ déborde sur le 3, qui n'est pas dans $E$ : il ne sait pas.

`K_1(E) = {0, 1, 2}` : le joueur 1 est mieux informé sur cette partie du monde, il sait $E$ partout sauf au monde 3.

Les deux opérateurs diffèrent, et c'est tout l'objet du carnet : **la connaissance dépend de l'information, pas du monde seul**. Un même événement peut être connu d'un joueur et ignoré de l'autre au même monde.

### Exercice 1 — vérifier les axiomes S5 sur un modèle que vous construisez

Les trois axiomes S5 se vérifient par énumération exhaustive : pour tout monde et tout événement, les trois inclusions doivent tenir. Écrivez la fonction `check_s5(model)` qui renvoie un dictionnaire `{axiome: bool}` pour `"veracite"`, `"intro_positive"`, `"intro_negative"`.

Deux pièges, tous deux dans la définition et non dans le code : (a) la véracité porte sur $K_i(E) \subseteq E$, pas sur $E \subseteq K_i(E)$ ; (b) l'introspection négative demande un événement qui est lui-même un ensemble de mondes — $\lnot K_i E$ est $\Omega \setminus K_i(E)$, pas une proposition sur la partition.

In [2]:
def check_s5(model: EpistemicModel) -> dict:
    """Verifie les trois axiomes S5 par enumeration exhaustive.

    Retourne {"veracite": bool, "intro_positive": bool, "intro_negative": bool}.
    """
    all_events = [frozenset(s) for r in range(len(model.worlds) + 1)
                  for s in combinations(sorted(model.worlds), r)]
    resultat = {"veracite": True, "intro_positive": True, "intro_negative": True}

    # TODO etudiant : boucler sur les evenements, les joueurs et les mondes.
    # Pour chaque axiome, mettre le drapeau a False des qu'un contre-exemple apparait.
    # Etape 1 : K_i(E) doit etre inclus dans E.
    # Etape 2 : K_i(E) doit etre inclus dans K_i(K_i(E)).
    # Etape 3 : Omega \ K_i(E) doit etre inclus dans K_i(Omega \ K_i(E)).

    return resultat


print("Exercice a completer : check_s5")
print("Rappel du resultat attendu sur le modele temoin : les trois axiomes sont vrais.")

Exercice a completer : check_s5
Rappel du resultat attendu sur le modele temoin : les trois axiomes sont vrais.


## 2. Connaissance mutuelle ou connaissance commune ?

Deux notions que le langage courant confond et que le calcul sépare :

- la **connaissance mutuelle** de $E$ : tout le monde sait $E$ — c'est $M(E) = \bigcap_i K_i(E)$ ;
- la **connaissance commune** de $E$ : tout le monde sait $E$, tout le monde sait que tout le monde sait $E$, et ainsi de suite — c'est le **plus grand point fixe** de l'opérateur $E \mapsto M(E) \cap E$.

Sur un monde fini, la suite $E_0 = E$, $E_{n+1} = M(E_n) \cap E_n$ est décroissante : elle se stabilise en au plus $|\Omega|$ étapes, et sa limite **est** $CK(E)$. Aucune infinité n'est nécessaire pour la calculer.

In [3]:
def mutual_knowledge(model: EpistemicModel, event) -> frozenset:
    """M(E) = intersection des K_i(E) sur tous les joueurs."""
    return frozenset.intersection(*[model.knows(i, event) for i in range(len(model.partitions))])


def common_knowledge(model: EpistemicModel, event):
    """CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite)."""
    current = frozenset(event)
    trace = [current]
    while True:
        nxt = mutual_knowledge(model, current) & current
        trace.append(nxt)
        if nxt == current:
            return current, trace
        current = nxt


mutual = mutual_knowledge(MODEL, E)
ck, trace = common_knowledge(MODEL, E)

print("connaissance mutuelle M(E) :", sorted(mutual))
print("suite des iteres          :", [sorted(t) for t in trace])
print("connaissance commune CK(E):", sorted(ck))
print("K_0 K_1(E)                :", sorted(MODEL.knows(0, MODEL.knows(1, E))))
print("K_1 K_0(E)                :", sorted(MODEL.knows(1, MODEL.knows(0, E))))

connaissance mutuelle M(E) : [0, 1]
suite des iteres          : [[0, 1, 2], [0, 1], [0], [], []]
connaissance commune CK(E): []
K_0 K_1(E)                : [0, 1]
K_1 K_0(E)                : [0]


### Lecture du résultat

La suite des itérés est `{0,1,2} -> {0,1} -> {0} -> {} -> {}` : elle se stabilise sur l'ensemble **vide** en trois pas. La connaissance commune de $E$ est donc vide — personne, dans aucun monde, ne sait que $E$ est de connaissance commune.

Et pourtant la connaissance mutuelle **n'est pas vide** : `M(E) = {0, 1}`. Aux mondes 0 et 1, les deux joueurs savent $E$. On peut même empiler un cran : `K_0 K_1(E) = {0, 1}` et `K_1 K_0(E) = {0}` — un niveau de raisonnement de plus tient encore.

C'est la distinction que le carnet veut rendre visible, et elle est **plus forte** que ce que la prose suggère d'habitude : la connaissance mutuelle, et même quelques niveaux de « je sais qu'il sait », peuvent tenir alors que la connaissance commune est vide. Le passage à l'infini n'est pas une formalité — il retire des mondes à chaque cran.

### Exercice 2 — trouver un modèle où la connaissance commune n'est pas vide

Construisez un modèle à trois mondes et un événement $E$ tels que $CK(E) = E$ et $E \neq \Omega$. Autrement dit : un événement qui est de connaissance commune sans être trivial.

Indice : il faut que $E$ soit une **réunion de cellules pour chaque joueur** — c'est exactement la condition d'auto-évidence (*self-evident event*) qui caractérise la connaissance commune. Vérifiez votre construction avec `common_knowledge`, puis expliquez en une phrase pourquoi l'événement trivial $\Omega$ est toujours de connaissance commune et pourquoi ce n'est pas un cas intéressant.

In [4]:
def modele_a_connaissance_commune():
    """Construit un modele a 3 mondes ou CK(E) = E, avec E different de Omega.

    Retourne (model, E).
    """
    # TODO etudiant : choisir les partitions pour que E soit une reunion de cellules
    # pour CHAQUE joueur, sans que E soit tout Omega.
    # Etape 1 : definir worlds = frozenset({0, 1, 2}).
    # Etape 2 : choisir partitions, par exemple un joueur qui distingue tout et un autre
    #           qui ne distingue pas deux mondes du meme cote de E.
    # Etape 3 : appeler common_knowledge et verifier CK(E) == E.
    print("Exercice a completer : modele_a_connaissance_commune")
    return None, None


print("Exercice a completer : construire le modele, puis afficher CK(E).")

Exercice a completer : construire le modele, puis afficher CK(E).


## 3. Agreeing to disagree — et le piège de la formalisation

Deux agents partagent un **prior commun** $\mu$ sur $\Omega$ et ont chacun leur information $P_1, P_2$. Le posterior du joueur $i$ au monde $\omega$ est

$$q_i(\omega) = \mu\big(E \mid P_i(\omega)\big)$$

**Théorème d'Aumann (1976).** Si les deux agents ont le même prior, il n'existe aucun monde où il soit de connaissance commune que $q_1 = a$ et $q_2 = b$ avec $a \neq b$.

Le carnet ne se contente pas de citer le théorème : il l'**éprouve par énumération exhaustive** sur tous les modèles à quatre mondes — les 15 partitions de $\Omega$, leurs 225 couples, et les 14 événements non triviaux. Soit 3150 modèles-événements.

La formalisation du test est le point délicat. « Ils sont en désaccord » est l'événement $D = \{\omega : q_1(\omega) \neq q_2(\omega)\}$. Ce n'est **pas** l'énoncé du théorème, qui porte sur la connaissance commune des **valeurs** : pour chaque couple $(a, b)$ avec $a \neq b$, l'événement $V_{a,b} = \{\omega : q_1(\omega) = a,\ q_2(\omega) = b\}$. Le code teste $V_{a,b}$, et la cellule suivante mesure aussi ce que donnerait le test naïf sur $D$.

In [5]:
def set_partitions(items):
    """Enumere toutes les partitions d'un ensemble fini (restricted growth strings)."""
    items = list(items)
    if not items:
        yield []
        return
    first, rest = items[0], items[1:]
    for part in set_partitions(rest):
        for i in range(len(part)):
            yield part[:i] + [[first] + part[i]] + part[i + 1:]
        yield [[first]] + part


def posterior(model: EpistemicModel, player: int, event) -> dict:
    """q_i(omega) = mu(E | cellule du joueur en omega)."""
    event = frozenset(event)
    out = {}
    for w in model.worlds:
        block = model.cell(player, w)
        masse = sum(model.prior[x] for x in block)
        out[w] = sum(model.prior[x] for x in block & event) / masse
    return out


def balayage_aumann(worlds, prior):
    """Enumere tous les couples de partitions et tous les evenements non triviaux.

    Compte (1) les paires de valeurs en desaccord examinees, (2) celles dont le
    desaccord est de connaissance commune -- le theoreme d'Aumann en attend zero,
    (3) ce que rendrait le test naif sur l'evenement D = {q1 != q2}.
    """
    partitions = [tuple(frozenset(b) for b in p) for p in set_partitions(sorted(worlds))]
    evenements = [frozenset(s) for r in range(1, len(worlds))
                  for s in combinations(sorted(worlds), r)]
    paires_examinees = 0
    violations = 0
    test_naif = 0
    for pa in partitions:
        for pb in partitions:
            modele = EpistemicModel(worlds=frozenset(worlds), partitions=(pa, pb), prior=prior)
            for ev in evenements:
                qa, qb = posterior(modele, 0, ev), posterior(modele, 1, ev)
                deja_vus = set()
                for w in sorted(worlds):
                    cle = (round(qa[w], 12), round(qb[w], 12))
                    if cle in deja_vus:
                        continue
                    deja_vus.add(cle)
                    if qa[w] == qb[w]:
                        continue
                    evenement_valeurs = frozenset(x for x in worlds
                                                 if (round(qa[x], 12), round(qb[x], 12)) == cle)
                    paires_examinees += 1
                    if w in common_knowledge(modele, evenement_valeurs)[0]:
                        violations += 1
                D = frozenset(x for x in worlds if qa[x] != qb[x])
                ckD = common_knowledge(modele, D)[0]
                test_naif += len(D & ckD)
    return len(partitions), len(evenements), paires_examinees, violations, test_naif


n_part, n_ev, paires, violations, naif = balayage_aumann(
    OMEGA, {w: 0.25 for w in OMEGA}
)
print("partitions de 4 mondes              :", n_part)
print("evenements non triviaux             :", n_ev)
print("modeles-evenements balayes          :", n_part ** 2 * n_ev)
print("paires de valeurs en desaccord      :", paires)
print("violations d'Aumann (attendu : 0)   :", violations)
print("test naif sur D = {q1 != q2}        :", naif, "cas de desaccord de connaissance commune")

partitions de 4 mondes              : 15
evenements non triviaux             : 14
modeles-evenements balayes          : 3150
paires de valeurs en desaccord      : 5632
violations d'Aumann (attendu : 0)   : 0
test naif sur D = {q1 != q2}        : 4832 cas de desaccord de connaissance commune


### Lecture du résultat

Le test correct rend **0 violation** sur les 5632 paires de valeurs en désaccord examinées : sous prior commun, aucun couple de posteriors distincts n'est jamais de connaissance commune. Le théorème tient sur l'énumération complète, pas sur un exemple choisi.

Le test naïf, lui, rend **4832 cas** de désaccord de connaissance commune. Ce n'est pas une réfutation d'Aumann : c'est la mesure de l'écart entre deux événements différents. Que les agents sachent qu'ils **ne sont pas d'accord** n'empêche rien — ils peuvent parfaitement le savoir, et même le savoir mutuellement, dans un modèle où chacun interprète l'information différemment. Ce que le théorème interdit, c'est qu'ils sachent **sur quelles valeurs précises** ils diffèrent, parce que cette connaissance-là, jointe au prior commun, permettrait de recalculer la moyenne des deux posteriors et de forcer l'égalité.

Un carnet qui testerait $D$ conclurait « Aumann est faux » et aurait tort. La formalisation est ici tout le contenu, et le nombre 4832 est là pour qu'on s'en souvienne.

In [6]:
# Controle negatif : deux agents aux priors DIFFERENTS.
# Les partitions sont triviales des deux cotes : chacun sait tout, donc le desaccord
# sur les valeurs est de connaissance commune partout -- et c'est coherent avec Aumann,
# qui suppose le prior commun.
PRIOR_1 = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}
PRIOR_2 = {0: 0.70, 1: 0.10, 2: 0.10, 3: 0.10}
E_CTRL = frozenset({0})

modele_1 = EpistemicModel(worlds=OMEGA, partitions=((OMEGA,), (OMEGA,)), prior=PRIOR_1)
modele_2 = EpistemicModel(worlds=OMEGA, partitions=((OMEGA,), (OMEGA,)), prior=PRIOR_2)

q1 = posterior(modele_1, 0, E_CTRL)
q2 = posterior(modele_2, 0, E_CTRL)

# On mesure les deux joueurs du meme modele mais sous des priors differents :
# chacun croit le monde entier possible, l'ecart est donc identique partout.
ecart = {w: round(q1[w], 4) for w in OMEGA}
print("prior uniforme  -> q = ", ecart[0])
print("prior biaise    -> q = ", round(sum(PRIOR_2[x] for x in OMEGA & E_CTRL), 4))
evenement_ecart = frozenset(w for w in OMEGA if abs(q1[w] - q2[w]) > 1e-9)
ck_ecart = common_knowledge(
    EpistemicModel(worlds=OMEGA, partitions=((OMEGA,), (OMEGA,)), prior=PRIOR_1),
    evenement_ecart,
)[0]
print("desaccord             :", sorted(evenement_ecart))
print("CK(desaccord)         :", sorted(ck_ecart))

prior uniforme  -> q =  0.25
prior biaise    -> q =  0.7
desaccord             : [0, 1, 2, 3]
CK(desaccord)         : [0, 1, 2, 3]


### Lecture du contrôle négatif

Avec le même prior, deux agents qui savent tout calculent le même posterior : pas de désaccord. Avec des priors différents, l'écart existe et il est **de connaissance commune partout** — les deux partitions sont triviales, donc tout événement est auto-évident.

C'est exactement ce que le théorème autorise : il ne dit pas que le désaccord est impossible, il dit que **le prior commun l'interdit**. Retirer l'hypothèse du prior commun suffit à faire apparaître le désaccord de connaissance commune. Le contrôle négatif n'est donc pas décoratif : sans lui, un carnet qui ne trouverait aucun désaccord ne saurait pas distinguer « le théorème tient » de « le modèle est dégénéré ».

### Exercice 3 — trois agents, même théorème

Le théorème d'Aumann s'énonce pour deux agents ; avec trois agents qui partagent un prior, la conclusion tient encore. Vérifiez-le sur un seul modèle, sans balayage exhaustif : construisez trois partitions de $\{0,1,2,3\}$ et un événement $E$, calculez les trois posteriors, et cherchez un monde où les trois valeurs sont deux à deux distinctes. Montrez que l'événement « ces trois valeurs précises » n'est jamais de connaissance commune.

Indice : `common_knowledge` prend un modèle ; `EpistemicModel` accepte autant de partitions que vous voulez dans son tuple. Cherchez d'abord un désaccord entre trois agents, puis testez sa connaissance commune — le résultat intéressant est que le désaccord existe, abondamment, mais qu'il n'est jamais su.

In [7]:
def trois_agents_verification():
    """Cherche un desaccord entre trois agents sous prior commun, puis mesure sa CK."""
    # TODO etudiant : construire un EpistemicModel a trois partitions sur Omega,
    # calculer posterior(modele, i, E) pour i = 0, 1, 2,
    # trouver un monde ou les trois valeurs sont distinctes,
    # puis appeler common_knowledge sur l'evenement des valeurs exactes.
    print("Exercice a completer : trois_agents_verification")
    return None


print("Exercice a completer : trois agents sous prior commun.")

Exercice a completer : trois agents sous prior commun.


## 4. Ce que l'épistémique change pour l'élimination itérée

Reste à relier tout cela au palier 02. La rationalité d'un joueur est un événement du modèle : au monde $\omega$, le joueur $i$ est rationnel si sa stratégie maximise son espérance conditionnelle à sa cellule d'information. L'hypothèse **CKR** — *common knowledge of rationality* — est alors $CK(R)$ avec $R = \bigcap_i R_i$, et elle a une conséquence calculable sur les stratégies.

Sur un jeu fini, l'élimination itérée des stratégies strictement dominées (IESDS) produit exactement les stratégies qui survivent à la connaissance commune de la rationalité : chaque tour d'élimination est un cran de l'itération $M^n$ de la section 2. Le code ci-dessous fait tourner l'élimination sur un jeu $3 \times 3$ et affiche l'ensemble résiduel — le lien avec les sections précédentes est que l'ensemble des stratégies survivantes est un événement **auto-évident pour chaque joueur**, c'est-à-dire exactement la condition de connaissance commune.

In [8]:
JEU = {
    "T": {"L": (3, 3), "R": (1, 1)},
    "M": {"L": (2, 2), "R": (1, 1)},
    "B": {"L": (0, 0), "R": (0, 0)},
}


def eliminer_dominées(jeu: dict):
    """IESDS : retire a chaque tour les strategies strictement dominees des deux joueurs."""
    lignes = list(jeu)                       # joueur ligne
    colonnes = list(jeu[lignes[0]])           # joueur colonne
    tours = []
    while True:
        retirer_l = []
        for l in lignes:
            for l2 in lignes:
                if l2 != l and all(jeu[l2][c][0] > jeu[l][c][0] for c in colonnes):
                    retirer_l.append(l)
                    break
        retirer_c = []
        for c in colonnes:
            for c2 in colonnes:
                if c2 != c and all(jeu[l][c2][1] > jeu[l][c][1] for l in lignes):
                    retirer_c.append(c)
                    break
        if not retirer_l and not retirer_c:
            return lignes, colonnes, tours
        lignes = [l for l in lignes if l not in retirer_l]
        colonnes = [c for c in colonnes if c not in retirer_c]
        tours.append((retirer_l, retirer_c))


restant_l, restant_c, historique = eliminer_dominées(JEU)
print("tours d'elimination :", historique)
print("lignes survivantes  :", restant_l)
print("colonnes survivantes:", restant_c)
print("nombre de tours     :", len(historique))

tours d'elimination : [(['B'], []), ([], ['R']), (['M'], [])]
lignes survivantes  : ['T']
colonnes survivantes: ['L']
nombre de tours     : 3


### Lecture du résultat

L'élimination retire `['B']`, puis `['R']`, puis `['M']` : trois tours, et l'ensemble survivant est le singleton `(['T'], ['L'])`. La cascade est instructive. Au premier tour, `B` est strictement dominée par `M` (2 > 0 et 1 > 0). Au deuxième, `R` devient strictement dominée par `L` — mais elle ne l'était **pas** avant, parce que sur la ligne `B` les deux colonnes rapportaient 0 : la dominance n'apparaît qu'après une première élimination. Au troisième tour, `M` tombe face à `T`. C'est cette dépendance en chaîne qui fait de l'élimination un calcul **itératif** et non une simple passe, et c'est elle que la connaissance commune de la rationalité autorise : chaque tour suppose que l'adversaire a déjà raisonné le tour précédent.

Ce qui compte pour ce carnet n'est pas le résultat du jeu mais la **forme** du calcul : chaque tour d'élimination suppose que le joueur sait que son adversaire est rationnel, donc retire une stratégie, ce qui rend une nouvelle stratégie dominée au tour suivant. C'est la suite $M^n$ de la section 2 appliquée à l'événement de rationalité. La connaissance commune n'est donc pas un ornement philosophique du palier 02 : c'est ce qui donne un sens à « éliminer itérativement » plutôt qu'à « éliminer une fois ».

## Synthèse

Trois choses ont été **calculées** ici, et non citées :

| Résultat | Comment il est obtenu | Ce que le carnet ajoute |
|---|---|---|
| Les axiomes S5 tiennent sur un modèle à partitions | énumération de tous les événements et de tous les mondes | la connaissance est dérivée de l'information, pas postulée |
| La connaissance mutuelle peut tenir alors que la connaissance commune est vide | itération jusqu'au point fixe, avec la suite des itérés affichée | la distinction n'est pas graduelle, elle est de nature |
| Sous prior commun, un désaccord de posteriors n'est jamais de connaissance commune | énumération exhaustive : 15 partitions, 225 couples, 14 événements | 5632 paires examinées, 0 violation — et 4832 faux contre-exemples si l'on teste la mauvaise formalisation |

## Suite

Les chapitres suivants de Bonanno (10 à 12 : équilibre séquentiel faible, équilibre séquentiel, équilibre bayésien parfait) ne sont pas traités ici, et la liste d'écarts de la série le dit : `weak sequential` et « équilibre bayésien parfait » ne comptent aucune occurrence dans les carnets. Ce sont eux qui utilisent la machinerie construite ici — les **croyances hors équilibre** sont exactement des cellules d'information auxquelles on affecte un posterior, et la cohérence d'une évaluation est une condition sur la façon dont ces posteriors se déduisent les uns des autres.

## Références

- Bonanno, G. (2015). *Game Theory* — ch. 7 *Knowledge and Common Knowledge*, ch. 8 *Beliefs*, ch. 9 *Common Knowledge of Rationality*. Licence CC BY-NC-ND 4.0 (page de copyright du PDF archivé).
- Aumann, R. J. (1976). « Agreeing to Disagree ». *The Annals of Statistics* 4(6), 1236-1239 — l'énoncé du théorème et sa démonstration par martingales.
- Geanakoplos, J. (1992). « Common Knowledge ». *Journal of Economic Perspectives* 6(4), 53-82 — la revue de référence sur les modèles à partitions et leurs limites.
- Halpern, J. Y. & Moses, Y. (1992). « A Guide to Completeness and Complexity for Modal Logics of Knowledge and Belief ». *Artificial Intelligence* 54, 319-379 — la variante probabiliste de S5 utilisée ici.